In [3]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

load_dotenv(PROJECT_ROOT / ".env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER") or os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

assert NEO4J_URI is not None, "NEO4J_URI is missing"
assert NEO4J_USER is not None, "NEO4J_USER is missing"
assert NEO4J_PASSWORD is not None, "NEO4J_PASSWORD is missing"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver

In [5]:
def run_query(cypher, params=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(cypher, params or {})
        return pd.DataFrame([dict(record) for record in result])

In [6]:
run_query("""
MATCH (n)
RETURN labels(n) AS labels, count(n) AS count
ORDER BY count DESC
""")

,labels,count
0,[Recording],70
1,[Tag],45
2,[Person],44
3,[Artist],6
4,[ReleaseGroup],6
5,[Release],6


In [7]:
run_query("""
MATCH ()-[r]->()
RETURN type(r) AS relationship_type, count(r) AS count
ORDER BY count DESC
""")

,relationship_type,count
0,HAS_TAG,75
1,HAS_TRACK,70
2,MEMBER_OF,52
3,CREATED,6
4,HAS_RELEASE,6


In [9]:
artists_df = run_query("""
MATCH (a:Artist)
RETURN coalesce(a.name, a.artist_name) AS artist_name
ORDER BY artist_name
""")

artists_df

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `artist_name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=27, offset=44>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 44, 'line': 3, 'column': 27}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (a:Artist)\nRETURN coalesce(a.name, a.artist_name) AS artist_name\nORDER BY artist_name\n'


,artist_name
0,Blur
1,Deftones
2,Nirvana
3,Pearl Jam
4,Soundgarden
5,The Smashing Pumpkins


In [10]:
shared_member_query = """
MATCH (query:Artist)
WHERE coalesce(query.name, query.artist_name) = $query_artist

MATCH (query)<-[:MEMBER_OF]-(person:Person)-[:MEMBER_OF]->(candidate:Artist)
WHERE candidate <> query

WITH
    candidate,
    collect(DISTINCT coalesce(person.name, person.person_name)) AS shared_members

RETURN
    coalesce(candidate.name, candidate.artist_name) AS retrieved_artist,
    size(shared_members) AS graph_score,
    shared_members AS evidence

ORDER BY graph_score DESC, retrieved_artist
LIMIT $top_k
"""

In [11]:
def retrieve_by_shared_members(query_artist, top_k=3):
    results = run_query(
        shared_member_query,
        {
            "query_artist": query_artist,
            "top_k": top_k
        }
    )

    if results.empty:
        return pd.DataFrame(
            columns=[
                "query_artist",
                "rank",
                "retrieved_artist",
                "graph_score",
                "evidence_type",
                "evidence"
            ]
        )

    results.insert(0, "query_artist", query_artist)
    results.insert(1, "rank", range(1, len(results) + 1))
    results.insert(4, "evidence_type", "shared_member")

    return results[
        [
            "query_artist",
            "rank",
            "retrieved_artist",
            "graph_score",
            "evidence_type",
            "evidence"
        ]
    ]

In [12]:
retrieve_by_shared_members("Pearl Jam", top_k=3)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `artist_name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=34, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 3, 'column': 34}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (query:Artist)\nWHERE coalesce(query.name, query.artist_name) = $query_artist\n\nMATCH (query)<-[:MEMBER_OF]-(person:Person)-[:MEMBER_OF]->(candidate:Artist)\nWHERE candidate <> query\n\nWITH\n    candidate,\n    collect(DISTINCT coalesce(person.name, person.person_name)) AS shared_members\n\nRETURN\n 

,query_artist,rank,retrieved_artist,graph_score,evidence_type,evidence
0,Pearl Jam,1,Soundgarden,1,shared_member,[Matt Cameron]


In [13]:
retrieve_by_shared_members("Nirvana", top_k=3)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `artist_name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=34, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 3, 'column': 34}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (query:Artist)\nWHERE coalesce(query.name, query.artist_name) = $query_artist\n\nMATCH (query)<-[:MEMBER_OF]-(person:Person)-[:MEMBER_OF]->(candidate:Artist)\nWHERE candidate <> query\n\nWITH\n    candidate,\n    collect(DISTINCT coalesce(person.name, person.person_name)) AS shared_members\n\nRETURN\n 

,query_artist,rank,retrieved_artist,graph_score,evidence_type,evidence
0,Nirvana,1,Soundgarden,1,shared_member,[Jason Everman]


In [14]:
shared_tag_query = """
MATCH (query:Artist)
WHERE coalesce(query.name, query.artist_name) = $query_artist

MATCH (query)-[qt:HAS_TAG]->(tag:Tag)<-[ct:HAS_TAG]-(candidate:Artist)
WHERE candidate <> query

WITH
    candidate,
    collect(DISTINCT coalesce(tag.name, tag.tag_name)) AS shared_tags,
    count(DISTINCT tag) AS shared_tag_count,
    sum(coalesce(qt.count, 1) + coalesce(ct.count, 1)) AS tag_weight_score

RETURN
    coalesce(candidate.name, candidate.artist_name) AS retrieved_artist,
    shared_tag_count AS graph_score,
    tag_weight_score,
    shared_tags AS evidence

ORDER BY graph_score DESC, tag_weight_score DESC, retrieved_artist
LIMIT $top_k
"""

In [15]:
def retrieve_by_shared_tags(query_artist, top_k=3):
    results = run_query(
        shared_tag_query,
        {
            "query_artist": query_artist,
            "top_k": top_k
        }
    )

    if results.empty:
        return pd.DataFrame(
            columns=[
                "query_artist",
                "rank",
                "retrieved_artist",
                "graph_score",
                "evidence_type",
                "evidence"
            ]
        )

    results.insert(0, "query_artist", query_artist)
    results.insert(1, "rank", range(1, len(results) + 1))
    results.insert(4, "evidence_type", "shared_tag")

    return results[
        [
            "query_artist",
            "rank",
            "retrieved_artist",
            "graph_score",
            "evidence_type",
            "evidence"
        ]
    ]

In [16]:
retrieve_by_shared_tags("Pearl Jam", top_k=3)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `artist_name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=34, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 3, 'column': 34}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (query:Artist)\nWHERE coalesce(query.name, query.artist_name) = $query_artist\n\nMATCH (query)-[qt:HAS_TAG]->(tag:Tag)<-[ct:HAS_TAG]-(candidate:Artist)\nWHERE candidate <> query\n\nWITH\n    candidate,\n    collect(DISTINCT coalesce(tag.name, tag.tag_name)) AS shared_tags,\n    count(DISTINCT tag) AS s

,query_artist,rank,retrieved_artist,graph_score,evidence_type,evidence
0,Pearl Jam,1,Nirvana,7,shared_tag,"[alternative rock, grunge, usa, acoustic rock,..."
1,Pearl Jam,2,Blur,7,shared_tag,"[alternative rock, grunge, rock, art rock, exp..."
2,Pearl Jam,3,Soundgarden,4,shared_tag,"[alternative rock, grunge, hard rock, usa]"


In [17]:
all_graph_results = []

for artist in artists_df["artist_name"]:
    member_results = retrieve_by_shared_members(artist, top_k=3)
    tag_results = retrieve_by_shared_tags(artist, top_k=3)

    all_graph_results.append(member_results)
    all_graph_results.append(tag_results)

graph_results = pd.concat(all_graph_results, ignore_index=True)

graph_results

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `artist_name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=34, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 3, 'column': 34}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (query:Artist)\nWHERE coalesce(query.name, query.artist_name) = $query_artist\n\nMATCH (query)<-[:MEMBER_OF]-(person:Person)-[:MEMBER_OF]->(candidate:Artist)\nWHERE candidate <> query\n\nWITH\n    candidate,\n    collect(DISTINCT coalesce(person.name, person.person_name)) AS shared_members\n\nRETURN\n 

,query_artist,rank,retrieved_artist,graph_score,evidence_type,evidence
0,Blur,1,Pearl Jam,7,shared_tag,"[alternative rock, grunge, rock, art rock, exp..."
1,Blur,2,The Smashing Pumpkins,5,shared_tag,"[alternative rock, grunge, rock, shoegaze, neo..."
2,Blur,3,Nirvana,4,shared_tag,"[alternative rock, grunge, noise rock, rock]"
3,Deftones,1,Soundgarden,2,shared_tag,"[alternative metal, alternative rock]"
4,Deftones,2,Nirvana,2,shared_tag,"[alternative metal, alternative rock]"
5,Deftones,3,The Smashing Pumpkins,2,shared_tag,"[alternative rock, shoegaze]"
6,Nirvana,1,Soundgarden,1,shared_member,[Jason Everman]
7,Nirvana,1,Pearl Jam,7,shared_tag,"[alternative rock, grunge, usa, acoustic rock,..."
8,Nirvana,2,Soundgarden,6,shared_tag,"[alternative metal, alternative rock, american..."
9,Nirvana,3,Blur,4,shared_tag,"[alternative rock, grunge, noise rock, rock]"


In [18]:
out_path = PROJECT_ROOT / "data" / "processed" / "graph_retrieval_results.csv"

graph_results.to_csv(out_path, index=False)

out_path

WindowsPath('c:/uni/seriousism/reminisGraph/data/processed/graph_retrieval_results.csv')

In [19]:
pd.read_csv(out_path).head()

,query_artist,rank,retrieved_artist,graph_score,evidence_type,evidence
0,Blur,1,Pearl Jam,7,shared_tag,"['alternative rock', 'grunge', 'rock', 'art ro..."
1,Blur,2,The Smashing Pumpkins,5,shared_tag,"['alternative rock', 'grunge', 'rock', 'shoega..."
2,Blur,3,Nirvana,4,shared_tag,"['alternative rock', 'grunge', 'noise rock', '..."
3,Deftones,1,Soundgarden,2,shared_tag,"['alternative metal', 'alternative rock']"
4,Deftones,2,Nirvana,2,shared_tag,"['alternative metal', 'alternative rock']"


## Phase 8 Conclusion

This notebook builds the first graph retrieval baseline using Neo4j traversal.

The graph retriever retrieves candidate artists using two types of explicit MusicBrainz graph relationships:

1. **Shared members**, where two artists are connected through the same person.
2. **Shared tags**, where two artists are connected through overlapping MusicBrainz tags.

Shared-member retrieval is especially important because it can reveal historical or relational artist connections that may not appear strongly in vector similarity results. For example, artist connections through members provide graph-specific evidence beyond tag or album-text similarity.

Shared-tag retrieval is also useful as a graph-based metadata signal, but it may overlap more with the vector baseline because the vector embeddings also use tag-based artist text.

The output is saved as `graph_retrieval_results.csv` and will be compared against the vector retrieval baseline in the next phase.